In [ ]:
Content
1. Database Engine
2. Database Types
3. Database Cursors
4. Database Security
5. Database Best practices

# 1. Database Engine

In [ ]:
Database Engines
    - A database engine (also called a storage engine) is the core software component of a database management system (DBMS) that is responsible for storing, retrieving, and managing data on disk (or in memory).
    - the part that knows how to write data efficiently to storage, keep it safe, and get it back fast.
    - Responsibilities of a Database Engine
    	•	Data storage (on disk or in memory)
    	•	Index management
    	•	Query execution (reading/writing rows)
    	•	Transaction management (ACID compliance)
    	•	Concurrency control (locks, MVCC, isolation levels)
    	•	Crash recovery (redo logs, checkpoints)
        •	These library take cares of disk storage and CRUD
            - Can be simple as key-value store
            - or as complex as ACID and transaction with forign keys

# How database Engine works
    1. When you write a SQL query, the sql layers parses and plan the query
    2. The storage Engine (like InnoDB) retrieves the row from disk or cache.
    3. If updating, it also handles locking, redo logs, crash recovery.


# 2. Database Types

In [ ]:
# Types of DB

+---------------------+----------------------------------------------+----------------------------------------------+
| Data Model          | Description                                  | Examples                                     |
+---------------------+----------------------------------------------+----------------------------------------------+
| Relational (RDBMS)  | Stores data in structured tables, supports   | MySQL (InnoDB), PostgreSQL, Oracle DB,       |
|                     | SQL, enforces schema, ACID transactions      | Microsoft SQL Server                         |
+---------------------+----------------------------------------------+----------------------------------------------+
| Document-oriented   | Stores semi-structured data as JSON/BSON     | MongoDB (WiredTiger), CouchDB, ArangoDB,     |
|                     | documents, flexible schema                   | RavenDB                                      |
+---------------------+----------------------------------------------+----------------------------------------------+
| Key-Value Store     | Simple key → value lookups, ultra-fast       | Redis, DynamoDB, RocksDB, LevelDB,           |
|                     | access, often in-memory                      | Aerospike                                    |
+---------------------+----------------------------------------------+----------------------------------------------+
| Wide Column Store   | Stores data in column families (rows can     | Apache Cassandra, HBase, ScyllaDB,           |
|                     | have dynamic columns), good for large-scale  | Google Bigtable                              |
|                     | analytics                                    |                                              |
+---------------------+----------------------------------------------+----------------------------------------------+
| Graph Database      | Optimized for nodes and edges, supports      | Neo4j, JanusGraph, Amazon Neptune,           |
|                     | graph traversal queries (relationships)      | OrientDB                                     |
+---------------------+----------------------------------------------+----------------------------------------------+
| Time-Series DB      | Optimized for timestamped data with          | InfluxDB, TimescaleDB, QuestDB,              |
|                     | efficient range queries and retention        | Prometheus                                   |
+---------------------+----------------------------------------------+----------------------------------------------+
| Search Engine DB    | Designed for full-text search and ranking    | Elasticsearch, Solr, OpenSearch              |
+---------------------+----------------------------------------------+----------------------------------------------+
| Multi-Model DB      | Supports multiple data models (document,     | ArangoDB, Cosmos DB, OrientDB                |
|                     | graph, key-value) in one engine              |                                              |
+---------------------+----------------------------------------------+----------------------------------------------+

# 3. Database Cursors

In [ ]:
A cursor is a database object that lets you retrieve query results row-by-row instead of all at once.
Think of it like a pointer that “walks” through a result set.
	•	Without a cursor → You run a query and get the whole result set in one go.
	•	With a cursor → You process one row (or a small batch) at a time.

# Why Cursors Are Used
Cursors are helpful when:
	•	You need row-by-row processing that can’t be done easily with a single SQL statement.
	•	You’re performing procedural logic (loops, conditions) based on query results.
	•	You’re updating or deleting specific rows one by one while checking conditions.

Example situations:
	•	Migrating data with complex transformations.
	•	Sending notifications for each row.
	•	Calculating running totals where a single SQL query is hard to write.

Working:
    1.	Declare a cursor with a SELECT query.
	2.	Open the cursor (executes the SELECT, but does not fetch data yet).
	3.	Fetch rows one by one (or in batches) into variables.
	4.	Process each fetched row in procedural logic.
	5.	Close the cursor (release result set).
	6.	Deallocate the cursor (free memory/resources).

# Lifecycle
Declare → Open → Fetch → Process → Close → Deallocate               
               
# SQL Query
DECLARE @name NVARCHAR(50);

DECLARE employee_cursor CURSOR FOR
    SELECT Name FROM Employees WHERE Department = 'IT';

OPEN employee_cursor;

FETCH NEXT FROM employee_cursor INTO @name;

WHILE @@FETCH_STATUS = 0 #loop
BEGIN
    PRINT 'Processing employee: ' + @name;
    FETCH NEXT FROM employee_cursor INTO @name;
END;

CLOSE employee_cursor;
DEALLOCATE employee_cursor;      



# Pros & Cons
Pros:
	•	Handles row-by-row operations that pure SQL can’t do easily.
	•	Allows complex procedural logic.
    •	We can not pull the million rows in one go, so cursors are required in that case. 
Cons:
	•	Slow for large datasets (row-by-row is less efficient than set-based operations).
	•	Consumes more memory and server resources.

# Types
Type              | Movement/Behavior                              | Updates Visible? | Performance Impact | Typical Use Case
------------------|-----------------------------------------------|------------------|--------------------|------------------------------
Static            | Snapshot of result set; does not reflect DB   | No               | Medium             | Reporting where stable results are needed
                    changes after open.0
Dynamic           | Reflects changes in underlying data in realtime| Yes              | High               | Live dashboards or monitoring
Forward-only      | Moves only forward through rows                | Depends on type  | Low                | Simple one-pass processing
Scrollable        | Can move forward and backward through rows     | Depends on type  | Medium-High        | Complex navigation through data
Read-only         | Allows only reading data via cursor            | No               | Lower              | Data viewing without updates (Won’t block writes (only reads), so other APIs can usually update rows, unless your isolation level still locks them.)
Updatable         | Allows updating/deleting rows via cursor       | Yes              | Higher             | Row-by-row updates with conditions  (Might place row-level or page-level locks on the rows it’s pointing to, blocking updates until the cursor is closed or moves past that row.)      

In [ ]:
# Server side vs Client side cursor
	•	Server-side cursor = “The library keeps your book open for you, you come back and read a few pages at a time.”
	•	Client-side cursor = “You borrow the entire book and read it at home.”

1. Server-side cursor
	•	Used when you have millions of rows and can’t fetch all at once.
	•	Application requests N rows at a time (FETCH NEXT 100 ROWS) while cursor stays open on DB.
	•	Common in reporting tools, admin consoles, or APIs streaming partial data.
    •	Con: Keeps DB resources busy longer.
    - Key points:
    	•	Data stays on the server until fetched.
    	•	Keeps DB transaction open → can hold locks.
    	•	Useful for huge datasets.

# SQL Query
import psycopg2
conn = psycopg2.connect(
    dbname="testdb", user="user", password="pass", host="localhost"
)

# Create a server-side cursor by naming it
cur = conn.cursor(name="server_cursor") # Difference just add this name.

# Execute query - rows are not fully fetched yet
cur.execute("SELECT * FROM large_table")

# Fetch rows in chunks from the server
while True:
    rows = cur.fetchmany(100)  # only fetch 100 at a time
    if not rows:
        break
    for row in rows:
        print(row)

cur.close()
conn.close()

   
2. Client-Side Cursor Example
	•	Used when result set is small or moderate.
	•	DB sends all rows at once → stored in client memory → iteration happens locally.
	•	Many ORMs (e.g., Entity Framework with ToList() in .NET, Pandas in Python) effectively do this.
    - Key points:
    	•	Frees server resources right after query finishes.
    	•	Faster for small/medium datasets.
    	•	Not memory-friendly for very large datasets.   

# SQL Query
import psycopg2

conn = psycopg2.connect(
    dbname="testdb", user="user", password="pass", host="localhost"
)

# Default is client-side cursor
cur = conn.cursor()

cur.execute("SELECT * FROM small_table")

# Fetch all rows into client memory
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()                  

# 4. Database Security

In [ ]:
TLS/SSL in databases is about encrypting the network connection between your application and the database so data can’t be intercepted or tampered with in transit.
	•	Prevent man-in-the-middle attacks (MITM) → Without encryption, an attacker could intercept and modify queries/results.

When you enable TLS/SSL:
	1.	Handshake happens before queries
    	•	Client and server exchange certificates and negotiate encryption algorithms.
    	•	Client may validate the server’s certificate (and optionally the reverse if mutual TLS is used).
	2.	Data is encrypted in transit
    	•	Every packet sent between your app and DB is encrypted/decrypted.
	3.	Authentication can be stronger
    	•	Certificates can serve as an additional identity check beyond username/password.
	4.	Slight latency overhead
    	•	A few extra milliseconds for connection setup (handshake) and small per-query encryption/decryption cost.
	5.	Connection strings change
    	•	You typically add parameters like sslmode=require (Postgres), encrypt=true (SQL Server), or --ssl (MySQL CLI).
	6.	Certificates management
    	•	You must install, rotate, and trust correct CA-signed certs on both client and server.

# Connection string
# Without TLS
Server=tcp:mydb.database.windows.net,1433;Database=test;User Id=app;Password=secret;

# With TLS
Server=tcp:mydb.database.windows.net,1433;Database=test;User Id=app;Password=secret;Encrypt=true;TrustServerCertificate=false;

In [ ]:
# Homomorphic Encryption
    - Homomorphic encryption is an encryption method that allows computations to be performed directly on encrypted data, without needing to decrypt it first.
    - When the computation is complete, the result (still encrypted) can be sent back to the client and decrypted to get the correct answer — as if the computation had been done on plaintext.
    - Cons:
        - It is quite slow. 100–1000× slower than plaintext operations
    
    - It solves a big security problem
    	•	Normally, to query a DB, the server must decrypt your data → the server can see it.
    	•	With HE, the DB server can perform operations on ciphertext without ever learning the actual data.

# 5. Database Best practices

In [ ]:
1. How nulls are better than empty string
    - NULL can save space
    - For example:
    	•	VARCHAR(255) NULL with NULL → 0 bytes for value (just 1 bit in bitmap)
    	•	VARCHAR(255) with '' → at least 1–2 bytes.
    - Index Efficiency
    	•	NULL values are not stored in some index types (e.g., B-Tree in PostgreSQL unless NULLS FIRST/LAST is specified).
    	•	Smaller indexes → less disk I/O → faster lookups.
    	•	If you have a lot of “empty” values, storing them as NULL can shrink index size and improve performance.
    - Query Performance
    	•	If most rows are NULL, queries with IS NOT NULL can quickly skip them (bitmap filtering, partial index).
    	•	Empty strings will still be part of the index and scan, which is slower if you rarely need them.            
    - Storage Impact in MySQL InnoDB
    	•	Empty string: 10M × ~1 byte = ~10 MB extra (plus index cost)
    	•	NULL: Only 10M bits (~1.25 MB for NULL bitmap), no index entry for NULLs
            